# Experiment 2: Native 2-Channel Sentinel-1 Architecture (Ablation)

This notebook is a modified copy of `04_train_s1_with_tessa_baseline.ipynb`, created to run **Experiment 2** from `TODO_next_experiments.md` without touching the working baseline notebook.

## What this tests

The baseline notebook (04) feeds Tessa's `ConditionalUNet` a 4-channel condition per view by taking Sentinel-1's native VV/VH (2 channels) and repeating it: `[VV, VH, VV, VH]`. This was done purely to satisfy the model's constructor, which was built for Sentinel-2 RGB/NIR (4 real, distinct channels).

This notebook removes that repetition and trains a fresh model with `cond_channels = 2 * CONTEXT_K` instead of `4 * CONTEXT_K`, so the network only ever sees the real 2-channel SAR signal. Everything else -- data, split seed, attrs (still zero-filled), training loop, sampler, metrics -- is identical to notebook 04, so any difference in ZNCC/RMSE/JSD between this run and the baseline is attributable specifically to the channel-repetition adapter, not to a different experiment setup.

**Why this one first:** of the three ablations in the TODO, this is the cheapest to test (two code edits, no new data processing) and directly targets the most mechanically suspicious part of the adapter -- feeding a CNN duplicated channels it was never trained to expect is a plausible source of degraded feature extraction, independent of whether the underlying SAR signal is informative.

This notebook does not execute automatically. Run cells top to bottom on the GPU workstation.

## GPU configuration

This notebook requires a CUDA-enabled PyTorch installation. It fails early if CUDA is unavailable, so an accidental CPU training run cannot consume the full dataset silently.

In [ ]:
import os
import sys
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import rasterio
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), 'CUDA is required. Run this notebook on the GPU environment.'
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = True
print('GPU:', torch.cuda.get_device_name(0))

## Paths and training configuration

Same paths as notebook 04's final run (`lidar_patches_tuk_tessa` / `s1_patches_tuk_michel_provided`) so the data and split match the baseline exactly. If you trained the notebook-04 baseline on different paths, update `S1_DIR`/`LIDAR_DIR` here to match -- the comparison is only fair if both runs see the same patches and the same train/val split (same `SEED`, same shuffle).

`EXPERIMENT_TAG` isolates every output of this notebook (checkpoint, metrics JSON, plots) from notebook 04's baseline outputs, so nothing gets overwritten and you can diff the two directly afterward.

In [ ]:
WORKING_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche')
TESSA_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche/tessa_baseline')
REGION = 'tuk'
EXPERIMENT_TAG = 'native2ch'
S1_DIR = WORKING_REPO / 'input_data' / 's1_patches_tuk_michel_provided'
LIDAR_DIR = WORKING_REPO / 'input_data' / 'lidar_patches_tuk_tessa'
CHECKPOINT_DIR = WORKING_REPO / 'checkpoints'
OUTPUT_DIR = WORKING_REPO / 's1_training_outputs'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_NAME = f's1_{REGION}_{EXPERIMENT_TAG}_unet_best.pth'
METRICS_FILENAME = f's1_{EXPERIMENT_TAG}_validation_metrics.json'
CONTEXT_K = 3
TARGET_HW = (256, 256)
BATCH_SIZE = 8
EPOCHS = 100
TIMESTEPS = 1000
LEARNING_RATE = 1e-4
VAL_FRACTION = 0.15
SEED = 42
NOISE_SCHEDULE = 'linear'
ATTENTION_VARIANT = 'default'

## Import Tessa's baseline implementation

The working tree intentionally does not contain Tessa's model and metrics modules, so imports use the cloned baseline path explicitly.

In [ ]:
sys.path.insert(0, str(TESSA_REPO))
from src.model.unet import ConditionalUNet
from src.diffusion.scheduler import LinearDiffusionScheduler, CosineDiffusionScheduler
from src.diffusion.sampling import p_sample_loop_ddpm, p_sample_loop_ddim, p_sample_loop_plms
from src.utils.recon_metrics import rmse, bias, sigma_error, normal_angle_error, average_jsd_multiscale, log_psd_rmse, zncc

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

## Sentinel-1/LiDAR dataset adapter -- native 2-channel version

This is the one substantive change from notebook 04: the `sar_tensor.repeat(2, 1, 1)` line is removed, so each Sentinel-1 view contributes its real 2 channels (VV, VH in dB) instead of being duplicated to 4. `condition` is therefore `2 * CONTEXT_K` channels total, not `4 * CONTEXT_K`. The `attrs` vector stays zero-filled -- unchanged from the baseline -- since this experiment isolates the channel-repetition variable only.

In [ ]:
class LidarS1Dataset(Dataset):
    def __init__(self, s1_dir, lidar_dir, patch_ids, context_k=3, target_hw=(256, 256), train=False):
        self.s1_dir = Path(s1_dir)
        self.lidar_dir = Path(lidar_dir)
        self.patch_ids = list(patch_ids)
        self.context_k = context_k
        self.target_hw = target_hw
        self.train = train

    def __len__(self):
        return len(self.patch_ids)

    def __getitem__(self, index):
        patch_id = self.patch_ids[index]
        lidar_path = self.lidar_dir / f'lidar_patch_{patch_id}.tif'
        s1_path = self.s1_dir / f's1_patch_{patch_id}'
        with rasterio.open(lidar_path) as src:
            raw = src.read().astype(np.float32)
        target = raw[0]
        mask = (raw[1] > 0.5) if raw.shape[0] > 1 else np.isfinite(target)
        target = np.nan_to_num(target, nan=0.0, posinf=0.0, neginf=0.0)
        valid_count = max(1, int(mask.sum()))
        patch_mean = float(target[mask].sum() / valid_count)
        target = (target - patch_mean) * mask

        times = sorted(s1_path.glob('t*.tif'))[:self.context_k]
        if len(times) < self.context_k:
            raise RuntimeError(f'{s1_path} has fewer than {self.context_k} Sentinel-1 times')
        views = []
        for time_path in times:
            with rasterio.open(time_path) as src:
                sar = src.read()[:2].astype(np.float32)
            sar = np.nan_to_num(sar, nan=0.0, posinf=0.0, neginf=0.0)
            sar = np.maximum(sar, 1e-12)
            sar = 10.0 * np.log10(sar)
            sar_tensor = torch.from_numpy(sar).unsqueeze(0)
            sar_tensor = F.interpolate(sar_tensor, size=self.target_hw, mode='bilinear', align_corners=False).squeeze(0)
            views.append(sar_tensor)
        condition = torch.cat(views, dim=0)
        attrs = torch.zeros(8 * self.context_k, dtype=torch.float32)
        return {'lidar': torch.from_numpy(target).unsqueeze(0).float(), 'mask': torch.from_numpy(mask), 's1': condition.float(), 'attrs': attrs, 'patch_mean': torch.tensor(patch_mean), 'patch_id': patch_id}

## Pair patches and create a reproducible split

Same split logic, same `SEED`, as notebook 04 -- with matching `S1_DIR`/`LIDAR_DIR` above, this produces the identical train/val patch split, so the comparison between this run and the baseline isolates the architecture change.

In [ ]:
lidar_ids = {p.stem.split('_')[-1] for p in LIDAR_DIR.glob('lidar_patch_*.tif')}
s1_ids = {p.name.split('_')[-1] for p in S1_DIR.glob('s1_patch_*') if p.is_dir()}
paired_ids = sorted(lidar_ids & s1_ids)
assert paired_ids, 'No paired Sentinel-1/LiDAR patches found.'
random.Random(SEED).shuffle(paired_ids)
n_val = max(1, int(len(paired_ids) * VAL_FRACTION))
val_ids, train_ids = paired_ids[:n_val], paired_ids[n_val:]
print(f'Paired: {len(paired_ids)} | train: {len(train_ids)} | validation: {len(val_ids)}')
train_dataset = LidarS1Dataset(S1_DIR, LIDAR_DIR, train_ids, CONTEXT_K, TARGET_HW, train=True)
val_dataset = LidarS1Dataset(S1_DIR, LIDAR_DIR, val_ids, CONTEXT_K, TARGET_HW, train=False)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

## Initialize Tessa's ConditionalUNet and scheduler -- native 2-channel

`cond_channels=2 * CONTEXT_K` (was `4 * CONTEXT_K` in the baseline). `attr_dim` is unchanged at `8 * CONTEXT_K` since attrs are still zero-filled here (that swap is Experiment 1, not this one). This model starts from freshly initialized weights -- it is not weight-compatible with the notebook-04 checkpoint because the first convolution's input channel count differs.

In [ ]:
model = ConditionalUNet(in_channels=1, cond_channels=2 * CONTEXT_K, attr_dim=8 * CONTEXT_K, base_channels=128, embed_dim=256, unet_depth=4, attention_variant=ATTENTION_VARIANT, cond_k=CONTEXT_K).to(DEVICE)
scheduler = (LinearDiffusionScheduler(TIMESTEPS, device=DEVICE) if NOISE_SCHEDULE == 'linear' else CosineDiffusionScheduler(TIMESTEPS, device=DEVICE))
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## Training loop

Identical objective to notebook 04: sample a diffusion timestep, add forward noise to the demeaned LiDAR target, predict the clean target, optimize masked MSE. The only difference from the baseline loop is the checkpoint filename, which is tagged so it never overwrites the notebook-04 checkpoint. `num_workers=0` is used deliberately (not the `num_workers=4`/`persistent_workers=True` from the original 04 template) -- notebook 04 was fixed to `num_workers=0` after a CUDA-context/`fork()` DataLoader crash, so this copy starts from that already-fixed version rather than reintroducing the bug.

In [ ]:
def masked_mse(prediction, target, mask):
    valid = mask.bool().unsqueeze(1)
    error = (prediction - target) ** 2
    return error[valid].mean()

history = {'train_loss': [], 'val_loss': []}
best_val = float('inf')
for epoch in range(EPOCHS):
    model.train()
    train_total = 0.0
    for batch in train_loader:
        target = batch['lidar'].to(DEVICE, non_blocking=True)
        condition = batch['s1'].to(DEVICE, non_blocking=True)
        attrs = batch['attrs'].to(DEVICE, non_blocking=True)
        mask = batch['mask'].to(DEVICE, non_blocking=True)
        timestep = torch.randint(0, TIMESTEPS, (target.size(0),), device=DEVICE)
        noisy = scheduler.q_sample(target, timestep)
        prediction = model(noisy, condition, attrs, timestep)
        loss = masked_mse(prediction, target, mask)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        train_total += loss.item()
    model.eval()
    val_total = 0.0
    with torch.no_grad():
        for batch in val_loader:
            target = batch['lidar'].to(DEVICE, non_blocking=True)
            condition = batch['s1'].to(DEVICE, non_blocking=True)
            attrs = batch['attrs'].to(DEVICE, non_blocking=True)
            mask = batch['mask'].to(DEVICE, non_blocking=True)
            timestep = torch.randint(0, TIMESTEPS, (target.size(0),), device=DEVICE)
            prediction = model(scheduler.q_sample(target, timestep), condition, attrs, timestep)
            val_total += masked_mse(prediction, target, mask).item()
    train_loss = train_total / max(1, len(train_loader))
    val_loss = val_total / max(1, len(val_loader))
    history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
    print(f'Epoch {epoch + 1:03d}/{EPOCHS}: train={train_loss:.6f} val={val_loss:.6f}')
    if val_loss < best_val:
        best_val = val_loss
        torch.save({'model_state_dict': model.state_dict(), 'config': {'context_k': CONTEXT_K, 'timesteps': TIMESTEPS, 'noise_schedule': NOISE_SCHEDULE, 'region': REGION, 'experiment_tag': EXPERIMENT_TAG}, 'epoch': epoch + 1, 'val_loss': val_loss}, CHECKPOINT_DIR / CHECKPOINT_NAME)

## Evaluate with Tessa's reconstruction metrics

Loads the `native2ch` checkpoint saved above (not the notebook-04 baseline checkpoint). `condition` here has `2 * CONTEXT_K` channels, so `s1_condition` per example patch is shaped `(2 * CONTEXT_K, H, W)` -- each view contributes 2 channels (VV, VH), not 4. This matters for the reconstruction-grid cell below, which indexes into this tensor to pull out the VV band per view.

In [ ]:
best_path = CHECKPOINT_DIR / CHECKPOINT_NAME
checkpoint = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
sampler = p_sample_loop_ddim
metric_rows = []
example_patches = []  # keep a few full patches for the qualitative reconstruction grid
N_EXAMPLES = 6
with torch.no_grad():
    for batch in val_loader:
        target = batch['lidar'].to(DEVICE)
        condition = batch['s1'].to(DEVICE)
        attrs = batch['attrs'].to(DEVICE)
        mask = batch['mask'].to(DEVICE).bool()
        prediction = sampler(model, scheduler, target.shape, condition, attrs, DEVICE)
        means = batch['patch_mean'].to(DEVICE).view(-1, 1, 1, 1)
        gt_absolute = target + means
        pred_absolute = prediction + means
        for i, patch_id in enumerate(batch['patch_id']):
            gt_i, pred_i, mask_i = gt_absolute[i], pred_absolute[i], mask[i]
            gt_valid = gt_i.squeeze()[mask_i].cpu().numpy()
            pred_valid = pred_i.squeeze()[mask_i].cpu().numpy()
            row = {
                'patch_id': patch_id,
                'rmse_m': float(rmse(gt_i, pred_i, mask_i).item()),
                'bias_m': float(bias(gt_i, pred_i, mask_i).item()),
                'sigma_error_pct': float(sigma_error(gt_i, pred_i, mask_i).item()),
                'normal_angle_error_deg': float(normal_angle_error(gt_i, pred_i, mask_i, pixel_size=1.0, degrees=True).item()),
                'jsd': float(average_jsd_multiscale(gt_i, pred_i, pixel_size=1.0, mask=mask_i).item()),
                'psd_rmse': float(log_psd_rmse(gt_i, pred_i, pixel_size=1.0, mask=mask_i).item()),
                'zncc': float(zncc(gt_i, pred_i, mask_i).item()),
                'gt_std_val': float(gt_valid.std()) if gt_valid.size > 0 else float('nan'),
                'pred_std_val': float(pred_valid.std()) if pred_valid.size > 0 else float('nan'),
            }
            metric_rows.append(row)
            if len(example_patches) < N_EXAMPLES:
                example_patches.append({
                    'patch_id': patch_id,
                    'gt': gt_i.squeeze().cpu().numpy(),
                    'pred': pred_i.squeeze().cpu().numpy(),
                    'mask': mask_i.squeeze().cpu().numpy(),
                    's1_condition': condition[i].cpu().numpy(),  # shape (2*CONTEXT_K, H, W)
                })
metrics_path = OUTPUT_DIR / METRICS_FILENAME
with metrics_path.open('w') as handle:
    json.dump(metric_rows, handle, indent=2)
print('Saved:', metrics_path)
print('Mean metrics:', {key: float(np.nanmean([row[key] for row in metric_rows])) for key in metric_rows[0] if key != 'patch_id'})

## Per-metric violin plots (Tessa-style)

Same as notebook 04's violin-plot cell, saved under a tagged filename so it sits alongside the baseline plot instead of overwriting it.

In [ ]:
import pandas as pd
import seaborn as sns

df_val = pd.DataFrame(metric_rows)

metric_column_names = ['rmse_m', 'bias_m', 'sigma_error_pct', 'normal_angle_error_deg', 'zncc', 'jsd', 'psd_rmse']
metric_column_dict = {
    'rmse_m': 'RMSE (m)',
    'bias_m': 'Bias (m)',
    'sigma_error_pct': 'RMS Height Error (%)',
    'normal_angle_error_deg': 'Normal Angle Error (deg)',
    'zncc': 'Cross Correlation (ZNCC)',
    'jsd': 'Distribution Divergence (JSD)',
    'psd_rmse': 'Log Spectral Error (RMSE)',
}
metric_units_dict = {
    'rmse_m': ' m', 'bias_m': ' m', 'sigma_error_pct': '%', 'normal_angle_error_deg': '°',
    'zncc': '', 'jsd': '', 'psd_rmse': '',
}

fig, axes = plt.subplots(1, len(metric_column_names), figsize=(20, 6))
axes = axes.flatten()
colors = sns.color_palette('husl', len(metric_column_names))
for i, metric in enumerate(metric_column_names):
    sns.violinplot(y=df_val[metric], ax=axes[i], color=colors[i])
    axes[i].set_title(metric_column_dict[metric], fontsize=13)
    axes[i].set_ylabel('')
    mean_value = df_val[metric].mean()
    axes[i].annotate(f'Mean: {mean_value:.2f}{metric_units_dict[metric]}', xy=(0.5, -0.07), xycoords='axes fraction', ha='center', fontsize=16, color='black', fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f's1_{EXPERIMENT_TAG}_metric_violin_plots.png', dpi=150, bbox_inches='tight')
print('Saved:', OUTPUT_DIR / f's1_{EXPERIMENT_TAG}_metric_violin_plots.png')
plt.show()

## GT vs. predicted roughness scatter (Tessa-style)

Same as notebook 04's scatter cell, saved under a tagged filename.

In [ ]:
df_val = pd.DataFrame(metric_rows)

x = df_val['gt_std_val']
y = df_val['pred_std_val']

plt.figure(figsize=(6, 6))
plt.scatter(x, y, alpha=0.5)

mn, mx = np.nanmin([x, y]), np.nanmax([x, y])
plt.plot([mn, mx], [mn, mx], 'r--', label='1:1')

mask = ~np.isnan(x) & ~np.isnan(y)
z = np.polyfit(x[mask], y[mask], 1)
p = np.poly1d(z)
xs = np.array([mn, mx])
plt.plot(xs, p(xs), 'b-', label='Best fit')

r2 = np.corrcoef(x[mask], y[mask])[0, 1] ** 2
plt.text(0.05, 0.95, f'R²={r2:.3f}', transform=plt.gca().transAxes, va='top', bbox=dict(facecolor='white', alpha=0.7))

plt.xlabel('Ground truth')
plt.ylabel('Prediction')
plt.title(f'Validation LiDAR Residual Standard Deviation ({REGION}, {EXPERIMENT_TAG})')
plt.legend()
plt.gca().set_aspect('equal', adjustable='box')
plt.savefig(OUTPUT_DIR / f's1_{EXPERIMENT_TAG}_gt_pred_std_scatter.png', dpi=150, bbox_inches='tight')
print('Saved:', OUTPUT_DIR / f's1_{EXPERIMENT_TAG}_gt_pred_std_scatter.png')
plt.show()

## Qualitative reconstruction grid (Tessa-style)

Same layout as notebook 04, with one indexing change: since `s1_condition` now has `2 * CONTEXT_K` channels (no repetition), the VV band for view `k` is at index `k * 2`, not `k * 4`.

In [ ]:
from matplotlib.colors import SymLogNorm

n_show = min(6, len(example_patches))
num_context_rows = CONTEXT_K
num_data_rows = 4  # GT, Pred, Error, PDF
total_rows = num_context_rows + num_data_rows
num_cols = n_show

all_gt = np.stack([p['gt'] for p in example_patches[:n_show]])
all_pred = np.stack([p['pred'] for p in example_patches[:n_show]])
max_abs_resid = np.quantile(np.abs(np.concatenate([all_gt.ravel(), all_pred.ravel()])), 0.995)
signed_error = all_pred - all_gt
max_abs_error = np.quantile(np.abs(signed_error.ravel()), 0.995)
norm = SymLogNorm(linthresh=0.1, linscale=1.0, vmin=-max_abs_resid, vmax=max_abs_resid, base=10)

row_titles = [f'S1 VV view {k + 1}' for k in range(CONTEXT_K)] + ['GT LiDAR', 'Pred LiDAR', 'Error', 'Patch PDF']

tile_size_inches = 4.0
fig, axes = plt.subplots(total_rows, num_cols, figsize=(num_cols * tile_size_inches + 2, total_rows * tile_size_inches), squeeze=False)

for col in range(num_cols):
    p = example_patches[col]
    gt_i, pred_i, mask_i = p['gt'], p['pred'], p['mask']
    s1_cond = p['s1_condition']  # (2 * CONTEXT_K, H, W); channel 0 of each 2-channel block is VV (dB)

    axes[0, col].set_title(f"Patch {p['patch_id']}", fontsize=14, fontweight='bold')

    for k in range(CONTEXT_K):
        ax = axes[k, col]
        ax.imshow(s1_cond[k * 2], cmap='gray')
        ax.axis('off')

    row_gt = num_context_rows
    ax = axes[row_gt, col]
    im_gt = ax.imshow(gt_i, cmap='RdBu_r', norm=norm)
    ax.axis('off')

    row_pred = num_context_rows + 1
    ax = axes[row_pred, col]
    im_pred = ax.imshow(pred_i, cmap='RdBu_r', norm=norm)
    ax.axis('off')

    row_err = num_context_rows + 2
    err_i = pred_i - gt_i
    ax = axes[row_err, col]
    im_err = ax.imshow(err_i, cmap='seismic', vmin=-max_abs_error, vmax=max_abs_error)
    ax.axis('off')

    row_pdf = num_context_rows + 3
    ax = axes[row_pdf, col]
    gt_vals = gt_i[mask_i]
    pred_vals = pred_i[mask_i]
    if gt_vals.size > 1 and pred_vals.size > 1:
        all_vals = np.concatenate([gt_vals, pred_vals])
        vmin_pdf, vmax_pdf = np.quantile(all_vals, [0.01, 0.99])
        if np.isclose(vmin_pdf, vmax_pdf):
            vmin_pdf, vmax_pdf = all_vals.min(), all_vals.max()
        bins = np.linspace(vmin_pdf, vmax_pdf, 60)
        gt_pdf, edges = np.histogram(gt_vals, bins=bins, density=True)
        pred_pdf, _ = np.histogram(pred_vals, bins=bins, density=True)
        centers = 0.5 * (edges[:-1] + edges[1:])
        ax.plot(centers, gt_pdf, color='darkblue', lw=2, label='GT')
        ax.plot(centers, pred_pdf, color='red', lw=2, alpha=0.95, label='Pred')
        ax.fill_between(centers, gt_pdf, color='darkblue', alpha=0.15)
        ax.fill_between(centers, pred_pdf, color='red', alpha=0.15)
        ax.set_xlim(vmin_pdf, vmax_pdf)
    ax.set_xlabel('Residual (m)', fontsize=10)
    if col == 0:
        ax.set_ylabel('Density', fontsize=10)
        ax.legend(fontsize=10, frameon=False)
    else:
        ax.set_yticklabels([])
    ax.set_box_aspect(1)

for row in range(total_rows):
    axes[row, 0].text(-0.25, 0.5, row_titles[row], ha='right', va='center', transform=axes[row, 0].transAxes, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f's1_{EXPERIMENT_TAG}_patch_reconstructions_pdfs.png', dpi=200, bbox_inches='tight')
print('Saved:', OUTPUT_DIR / f's1_{EXPERIMENT_TAG}_patch_reconstructions_pdfs.png')
plt.show()

## Comparison protocol

Compare `s1_native2ch_validation_metrics.json` (this notebook) against `s1_validation_metrics.json` (notebook 04's baseline, zero-filled attrs + 4-channel repeated input) on the same patch split. Focus on `zncc` and `jsd` first -- these are where the baseline lagged Sentinel-2 most (ZNCC 0.166 vs Tessa's 0.74-0.78) while RMSE/bias were already comparable, so a real improvement here would show up as ZNCC rising and JSD falling relative to the baseline row in `CONCEPTS.md`.

If ZNCC improves substantially: the repeated-channel adapter was a real bottleneck, keep the native 2-channel architecture as the new baseline and layer Experiment 1 (real attrs) on top of it next.

If ZNCC barely moves: channel repetition wasn't the dominant issue, and Experiment 1 (real attrs) or Experiment 3 (despeckling) should be tried next, per the suggested order in `TODO_next_experiments.md`.